In [0]:
import os
from pyspark.sql import functions as F

BASE_DIR = '/Volumes/debora_ryan_susheela_hhs/default/upload_volume'
BRONZE_PARQUET_DIR = os.path.join(BASE_DIR, "bronze_output", "parquet_data_hhs")
os.makedirs(BRONZE_PARQUET_DIR, exist_ok=True)
print(f"BASE_DIR: {BASE_DIR}")
print(f"BRONZE_PARQUET_DIR: {BRONZE_PARQUET_DIR}")

In [0]:
# Reading parquet & create dataframe. 

from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import os
filepath_benefits = os.path.join(BRONZE_PARQUET_DIR, 'benefits')

filepath_rates = os.path.join(BRONZE_PARQUET_DIR, 'rates')

def read_parquet(filepath: str) -> DataFrame:
    data_f = spark.read.parquet(filepath)
    return data_f
    
df_benefits = read_parquet(filepath_benefits)
df_rates = read_parquet(filepath_rates)


In [0]:
df_rates.describe()

In [0]:
df_benefits.describe()

In [0]:
display(df_benefits.select("BenefitName").distinct().orderBy("BenefitName"))

In [0]:
display(df_benefits.select("IsCovered").distinct())

In [0]:
df_covered = df_benefits.filter(F.col("IsCovered") == "Covered")

In [0]:
df_grouped = (
    df_covered
    .groupBy("PlanId")
    .agg(
        F.concat_ws(",", F.collect_list("BenefitName")).alias("BenefitNames"),
        F.size(F.collect_list("BenefitName")).alias("BenefitCount")
    )
)

display(df_grouped)

In [0]:
df_grouped_transformed = (
    df_grouped
    .withColumn("PlanId", F.split(F.col("PlanId"), "-")[0])
    .dropDuplicates(["PlanId"])
)

display(df_grouped_transformed)

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def filter_baseline_rate(df: DataFrame) -> DataFrame:
    return (
        df
        .filter(
            (F.col("Age") == "30") &
            (F.col("Tobacco") == "No Preference")
        )
        .select(
            F.col("PlanId"),
            F.col("IndividualRate").cast("double").alias("IndividualRate")
        )
        .filter(F.col("IndividualRate").isNotNull())
        # .dropDuplicates(["PlanId"])
    )


df_rate_baseline = df_rates.transform(filter_baseline_rate)

display(df_rate_baseline)

In [0]:
df_joined_benefits_rates = df_rate_baseline.join(df_grouped_transformed, on="PlanId", how="inner")
display(df_joined_benefits_rates)

In [0]:
SILVER_PARQUET_DIR = os.path.join(BASE_DIR, "silver_output", "benefits_rates")

df_joined_benefits_rates.write.mode("overwrite").parquet(SILVER_PARQUET_DIR)

print(f"Silver layer saved to: {SILVER_PARQUET_DIR}")

# Unit Tests

In [0]:
# Install a few helpers we prepared for you
%pip uninstall -y databricks_helpers exercise_ev_databricks_unit_tests

# Install the databricks helpers 
# %pip install git+https://github.com/data-derp/databricks_helpers.git@sr/dbr_17.3_lts_testing
%pip install git+https://github.com/data-derp/databricks_helpers.git

# # Install the databricks test cases
# %pip install git+https://github.com/data-derp/exercise_ev_databricks_unit_tests.git@sr/dbr_17.3_lts_testing
%pip install git+https://github.com/data-derp/exercise_ev_databricks_unit_tests.git

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# ============================================================
# Test 1: filter_baseline_rate – keeps only Age=30, Tobacco=No Preference
# ============================================================
def test_filter_baseline_rate():
    schema = StructType([
        StructField("PlanId", StringType()),
        StructField("Age", StringType()),
        StructField("Tobacco", StringType()),
        StructField("IndividualRate", StringType()),
    ])
    data = [
        ("PLAN-A", "30", "No Preference", "100.50"),
        ("PLAN-B", "25", "No Preference", "80.00"),   # wrong age
        ("PLAN-C", "30", "Tobacco User", "120.00"),    # wrong tobacco
        ("PLAN-D", "30", "No Preference", None),       # null rate
        ("PLAN-E", "30", "No Preference", "200.00"),
    ]
    df = spark.createDataFrame(data, schema)
    result = df.transform(filter_baseline_rate)

    assert result.columns == ["PlanId", "IndividualRate"], f"Unexpected columns: {result.columns}"
    assert result.count() == 2, f"Expected 2 rows, got {result.count()}"
    plan_ids = [row.PlanId for row in result.collect()]
    assert "PLAN-A" in plan_ids and "PLAN-E" in plan_ids, f"Unexpected PlanIds: {plan_ids}"
    # Verify cast to double
    assert result.schema["IndividualRate"].dataType == DoubleType(), "IndividualRate should be DoubleType"
    print("✓ test_filter_baseline_rate PASSED")

test_filter_baseline_rate()

# ============================================================
# Test 2: IsCovered filter – keeps only "Covered" rows
# ============================================================
def test_filter_covered_benefits():
    schema = StructType([
        StructField("PlanId", StringType()),
        StructField("BenefitName", StringType()),
        StructField("IsCovered", StringType()),
    ])
    data = [
        ("PLAN-A", "Dental", "Covered"),
        ("PLAN-A", "Vision", "Not Covered"),
        ("PLAN-B", "Mental Health", "Covered"),
        ("PLAN-B", "Acupuncture", "Not Covered"),
    ]
    df = spark.createDataFrame(data, schema)
    result = df.filter(F.col("IsCovered") == "Covered")

    assert result.count() == 2, f"Expected 2 rows, got {result.count()}"
    benefit_names = sorted([row.BenefitName for row in result.collect()])
    assert benefit_names == ["Dental", "Mental Health"], f"Unexpected benefits: {benefit_names}"
    print("✓ test_filter_covered_benefits PASSED")

test_filter_covered_benefits()

# ============================================================
# Test 3: GroupBy aggregation – BenefitNames and BenefitCount
# ============================================================
def test_group_by_plan():
    schema = StructType([
        StructField("PlanId", StringType()),
        StructField("BenefitName", StringType()),
        StructField("IsCovered", StringType()),
    ])
    data = [
        ("PLAN-A", "Dental", "Covered"),
        ("PLAN-A", "Vision", "Covered"),
        ("PLAN-A", "Mental Health", "Covered"),
        ("PLAN-B", "Dental", "Covered"),
    ]
    df = spark.createDataFrame(data, schema)
    result = (
        df.groupBy("PlanId")
        .agg(
            F.concat_ws(",", F.collect_list("BenefitName")).alias("BenefitNames"),
            F.size(F.collect_list("BenefitName")).alias("BenefitCount")
        )
    )

    assert result.count() == 2, f"Expected 2 plans, got {result.count()}"
    plan_a = result.filter(F.col("PlanId") == "PLAN-A").collect()[0]
    assert plan_a.BenefitCount == 3, f"Expected 3 benefits for PLAN-A, got {plan_a.BenefitCount}"
    assert "," in plan_a.BenefitNames, "BenefitNames should be comma-separated"
    plan_b = result.filter(F.col("PlanId") == "PLAN-B").collect()[0]
    assert plan_b.BenefitCount == 1, f"Expected 1 benefit for PLAN-B, got {plan_b.BenefitCount}"
    print("✓ test_group_by_plan PASSED")

test_group_by_plan()

# ============================================================
# Test 4: PlanId transformation – split on "-" and dedup
# ============================================================
def test_plan_id_transformation():
    schema = StructType([
        StructField("PlanId", StringType()),
        StructField("BenefitNames", StringType()),
        StructField("BenefitCount", IntegerType()),
    ])
    data = [
        ("10046HI0020003-00", "Dental,Vision", 2),
        ("10046HI0020003-01", "Dental,Vision,Mental", 3),  # same prefix, different suffix
        ("10091OR0360004-01", "Dental", 1),
    ]
    df = spark.createDataFrame(data, schema)
    result = (
        df
        .withColumn("PlanId", F.split(F.col("PlanId"), "-")[0])
        .dropDuplicates(["PlanId"])
    )

    assert result.count() == 2, f"Expected 2 rows after dedup, got {result.count()}"
    plan_ids = sorted([row.PlanId for row in result.collect()])
    assert plan_ids == ["10046HI0020003", "10091OR0360004"], f"Unexpected PlanIds: {plan_ids}"
    # Verify suffix removed
    assert all("-" not in pid for pid in plan_ids), "PlanIds should not contain '-'"
    print("✓ test_plan_id_transformation PASSED")

test_plan_id_transformation()

# ============================================================
# Test 5: Inner join – only matching PlanIds survive
# ============================================================
def test_inner_join():
    rates_schema = StructType([
        StructField("PlanId", StringType()),
        StructField("IndividualRate", DoubleType()),
    ])
    benefits_schema = StructType([
        StructField("PlanId", StringType()),
        StructField("BenefitNames", StringType()),
        StructField("BenefitCount", IntegerType()),
    ])
    rates_data = [
        ("PLAN-A", 100.0),
        ("PLAN-B", 200.0),
        ("PLAN-C", 300.0),  # no matching benefit
    ]
    benefits_data = [
        ("PLAN-A", "Dental,Vision", 2),
        ("PLAN-B", "Mental Health", 1),
        ("PLAN-D", "Dental", 1),  # no matching rate
    ]
    df_rates_test = spark.createDataFrame(rates_data, rates_schema)
    df_benefits_test = spark.createDataFrame(benefits_data, benefits_schema)

    result = df_rates_test.join(df_benefits_test, on="PlanId", how="inner")

    assert result.count() == 2, f"Expected 2 rows from inner join, got {result.count()}"
    assert set(result.columns) == {"PlanId", "IndividualRate", "BenefitNames", "BenefitCount"}
    plan_ids = sorted([row.PlanId for row in result.collect()])
    assert plan_ids == ["PLAN-A", "PLAN-B"], f"Unexpected joined PlanIds: {plan_ids}"
    print("✓ test_inner_join PASSED")

test_inner_join()

print("\n" + "="*50)
print("ALL UNIT TESTS PASSED ✓")
print("="*50)